# 03 - Baseline Model Training

This notebook begins the **Modelling** phase of the CRISP-DM workflow.

The first model is a simple, explainable baseline for classifying rolled fingerprint images into four broad dermatoglyphic pattern classes:

- `arch`
- `left_slant_loop`
- `right_slant_loop`
- `whorl`

We start with a classical machine learning baseline using HOG image features and a class-weighted linear classifier. This gives us a professional benchmark before trying heavier deep learning models.

## How this notebook is structured

The notebook is designed to run section by section.

The main steps are:

1. Load the roll-only split.
2. Preview sample images.
3. Extract HOG features from the images.
4. Save the extracted features.
5. Train a baseline classifier.
6. Evaluate the model using academic metrics.

Each major output is saved so the notebook can be reopened without repeating every previous step.

## Install required libraries

Run this cell once if your environment does not already have the needed packages.

`scikit-image` is used for HOG features, and `scikit-learn` is used for the baseline classifier and evaluation.

In [ ]:
%pip install pandas numpy matplotlib pillow scikit-image scikit-learn joblib

## Section 1: Set up paths and imports

This cell keeps the project folders in one place.

It works whether the notebook is opened from the project root or from inside the `notebooks` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROCESSED_DIR / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Section 2: Load the roll-only split

This is the final dataset prepared in the previous notebook.

It contains only rolled fingerprint images and uses a subject-aware train, validation, and test split.

In [ ]:
split_path = PROCESSED_DIR / "roll_broad_model_split.csv"

id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

dataset = pd.read_csv(split_path, dtype=id_columns)

print("Rows:", len(dataset))
print("Subjects:", dataset["subject_id"].nunique())

In [ ]:
pd.crosstab(dataset["broad_class"], dataset["split"])[["train", "validation", "test"]]

## Section 3: Confirm image paths

Before extracting features, we check that the image paths point to real files.

This prevents silent training errors later.

In [ ]:
dataset["image_path"] = dataset["png_path"].apply(lambda path: PROJECT_ROOT / path)
dataset["image_exists"] = dataset["image_path"].apply(lambda path: path.exists())

dataset["image_exists"].value_counts()

In [ ]:
missing_images = dataset[dataset["image_exists"] == False]

if len(missing_images) > 0:
    raise FileNotFoundError(f"Missing image files: {len(missing_images)}")

print("All image files were found.")

## Section 4: Preview the modelling images

This visual check helps us confirm that the model will train on fingerprint images and that each class has visible examples.

In [ ]:
from PIL import Image

preview_groups = []

for class_name in sorted(dataset["broad_class"].unique()):
    class_rows = dataset[dataset["broad_class"] == class_name]
    sample_size = min(3, len(class_rows))
    preview_groups.append(class_rows.sample(sample_size, random_state=42))

preview_sample = pd.concat(preview_groups, ignore_index=True)

preview_sample[["broad_class", "subject_id", "finger_position", "png_path"]]

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(9, 10))

for axis, (_, row) in zip(axes.ravel(), preview_sample.iterrows()):
    image = Image.open(row["image_path"]).convert("L")
    axis.imshow(image, cmap="gray")
    axis.set_title(row["broad_class"])
    axis.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_training_image_preview.png", dpi=150)
plt.show()

## Section 5: Define image feature extraction

HOG means **Histogram of Oriented Gradients**.

In simple terms, it turns each fingerprint image into numbers that describe the direction and shape of ridge patterns. This is useful for a first baseline because fingerprint classes are strongly related to ridge flow.

In [ ]:
from skimage.feature import hog
from skimage.transform import resize

IMAGE_SIZE = (160, 160)

def load_grayscale_image(image_path):
    image = Image.open(image_path).convert("L")
    image = np.array(image) / 255.0
    return resize(image, IMAGE_SIZE, anti_aliasing=True)

In [ ]:
def extract_hog_features(image_path):
    image = load_grayscale_image(image_path)

    return hog(
        image,
        orientations=9,
        pixels_per_cell=(12, 12),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
    )

## Section 6: Extract and save HOG features

This step may take a little time because it reads every image and converts it into numeric features.

The output is saved to disk so model training can be repeated later without extracting features again.

In [ ]:
from tqdm.auto import tqdm

feature_rows = []

for image_path in tqdm(dataset["image_path"], desc="Extracting HOG features"):
    feature_rows.append(extract_hog_features(image_path))

features = np.vstack(feature_rows)

print("Feature matrix shape:", features.shape)

In [ ]:
label_names = sorted(dataset["broad_class"].unique())
label_to_id = {label: index for index, label in enumerate(label_names)}

labels = dataset["broad_class"].map(label_to_id).to_numpy()
splits = dataset["split"].to_numpy()

label_to_id

In [ ]:
feature_path = PROCESSED_DIR / "roll_hog_features.npz"

np.savez_compressed(
    feature_path,
    features=features,
    labels=labels,
    splits=splits,
    label_names=np.array(label_names),
)

print("Saved:", feature_path)

## Stop and review feature extraction

Pause here after running feature extraction.

Check that:

- all image files were found.
- the feature matrix has one row per image.
- `roll_hog_features.npz` was saved.

After this, the next cells can train and evaluate the baseline model without re-reading all images.

## Section 7: Load saved features for training

This section can run on its own after `roll_hog_features.npz` has been created.

In [ ]:
feature_file = np.load(PROCESSED_DIR / "roll_hog_features.npz", allow_pickle=True)

features = feature_file["features"]
labels = feature_file["labels"]
splits = feature_file["splits"]
label_names = feature_file["label_names"].tolist()

print("Features:", features.shape)
print("Labels:", label_names)

In [ ]:
train_mask = splits == "train"
validation_mask = splits == "validation"
test_mask = splits == "test"

X_train, y_train = features[train_mask], labels[train_mask]
X_validation, y_validation = features[validation_mask], labels[validation_mask]
X_test, y_test = features[test_mask], labels[test_mask]

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

## Section 8: Train the baseline classifier

We use a linear Support Vector Machine as the baseline classifier.

`class_weight="balanced"` helps the model pay more attention to the smaller `arch` class.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

baseline_model = make_pipeline(
    StandardScaler(),
    LinearSVC(class_weight="balanced", random_state=42, max_iter=5000),
)

baseline_model.fit(X_train, y_train)

print("Baseline model trained.")

In [ ]:
import joblib

model_path = MODEL_DIR / "hog_linear_svc_baseline.joblib"

joblib.dump(
    {
        "model": baseline_model,
        "label_names": label_names,
        "image_size": IMAGE_SIZE,
    },
    model_path,
)

print("Saved:", model_path)

## Section 9: Evaluate on validation data

Validation results help us understand whether the baseline is learning useful pattern differences before we touch the test set.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

validation_predictions = baseline_model.predict(X_validation)

validation_accuracy = accuracy_score(y_validation, validation_predictions)
print("Validation accuracy:", round(validation_accuracy, 4))

In [ ]:
print(classification_report(
    y_validation,
    validation_predictions,
    target_names=label_names,
))

## Section 10: Final test evaluation

The test set is used after the baseline model has been trained.

This gives the main academic result for the first model.

In [ ]:
test_predictions = baseline_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_predictions)
print("Test accuracy:", round(test_accuracy, 4))

In [ ]:
test_report = classification_report(
    y_test,
    test_predictions,
    target_names=label_names,
    output_dict=True,
)

test_report_table = pd.DataFrame(test_report).T
test_report_table.round(3)

## Section 11: Confusion matrix

The confusion matrix shows which fingerprint classes the model confuses with each other.

This is more useful than accuracy alone because it shows per-class weaknesses.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axis = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=label_names,
    xticks_rotation=25,
    cmap="Blues",
    ax=axis,
)

axis.set_title("Baseline Model Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_confusion_matrix.png", dpi=150)
plt.show()

## Section 12: Save evaluation results

The evaluation table is saved so it can be used later in the report.

In [ ]:
results_path = PROCESSED_DIR / "baseline_test_classification_report.csv"

test_report_table.to_csv(results_path)

print("Saved:", results_path)

## Stop and review the baseline model

Pause here after training and evaluation.

The key results to discuss are:

- validation accuracy
- test accuracy
- precision, recall, and F1-score for each class
- whether `arch` is weaker because it has fewer examples
- which classes are confused in the confusion matrix

After reviewing this baseline, we can decide whether to improve the model with better features, image preprocessing, or a CNN/transfer learning approach.

## Section 13: Error analysis setup

This section can run on its own after the baseline model has been trained.

The goal is to look beyond accuracy and understand what the model is getting wrong. This is important for an academic project because it explains the limitations of the baseline model.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROCESSED_DIR / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
feature_file = np.load(PROCESSED_DIR / "roll_hog_features.npz", allow_pickle=True)
model_bundle = joblib.load(MODEL_DIR / "hog_linear_svc_baseline.joblib")

features = feature_file["features"]
labels = feature_file["labels"]
splits = feature_file["splits"]
label_names = feature_file["label_names"].tolist()
baseline_model = model_bundle["model"]

print("Features:", features.shape)
print("Labels:", label_names)

## Section 14: Create a test prediction table

Here we attach the model predictions back to the original test rows.

This gives us a table showing the true class, predicted class, subject, finger position, and image path for each test image.

In [ ]:
id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

dataset = pd.read_csv(PROCESSED_DIR / "roll_broad_model_split.csv", dtype=id_columns)
dataset["image_path"] = dataset["png_path"].apply(lambda path: PROJECT_ROOT / path)

test_mask = splits == "test"
test_predictions = baseline_model.predict(features[test_mask])

test_results = dataset[dataset["split"] == "test"].copy().reset_index(drop=True)
test_results["true_label"] = [label_names[index] for index in labels[test_mask]]
test_results["predicted_label"] = [label_names[index] for index in test_predictions]
test_results["correct"] = test_results["true_label"] == test_results["predicted_label"]

test_results[["true_label", "predicted_label", "correct"]].head()

In [ ]:
prediction_path = PROCESSED_DIR / "baseline_test_predictions.csv"

test_results.to_csv(prediction_path, index=False)

print("Saved:", prediction_path)
print("Wrong predictions:", (test_results["correct"] == False).sum())

## Section 15: Identify the most common mistakes

This table shows the main confusion patterns.

For example, it can show whether arches are mostly being predicted as loops or whorls.

In [ ]:
wrong_results = test_results[test_results["correct"] == False].copy()

confusion_pairs = (
    wrong_results.groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs

In [ ]:
pair_plot_data = confusion_pairs.head(10).copy()
pair_plot_data["mistake"] = pair_plot_data["true_label"] + " -> " + pair_plot_data["predicted_label"]

ax = pair_plot_data.plot(kind="barh", x="mistake", y="count", figsize=(8, 5), legend=False)
ax.set_title("Most Common Baseline Mistakes")
ax.set_xlabel("Number of wrong predictions")
ax.set_ylabel("True class -> predicted class")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_common_mistakes.png", dpi=150)
plt.show()

## Section 16: View wrongly classified examples

The table and plot below show real test images that the model classified incorrectly.

This is useful for explaining whether the mistakes look visually difficult, noisy, or caused by similar ridge patterns.

In [ ]:
error_samples = []

for class_name in label_names:
    class_errors = wrong_results[wrong_results["true_label"] == class_name]
    sample_size = min(3, len(class_errors))

    if sample_size > 0:
        error_samples.append(class_errors.sample(sample_size, random_state=42))

error_sample = pd.concat(error_samples, ignore_index=True)

error_sample[["true_label", "predicted_label", "subject_id", "finger_position", "png_path"]]

In [ ]:
columns = 3
rows = int(np.ceil(len(error_sample) / columns))

fig, axes = plt.subplots(rows, columns, figsize=(10, rows * 3))
axes = np.array(axes).reshape(-1)

for axis, (_, row) in zip(axes, error_sample.iterrows()):
    image = Image.open(row["image_path"]).convert("L")
    axis.imshow(image, cmap="gray")
    axis.set_title(f"True: {row['true_label']}\nPred: {row['predicted_label']}")
    axis.axis("off")

for axis in axes[len(error_sample):]:
    axis.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "baseline_wrong_prediction_examples.png", dpi=150)
plt.show()

## Section 17: Baseline error interpretation

This final cell prints a short interpretation that can be adapted for the report.

In [ ]:
total_test_images = len(test_results)
total_wrong = len(wrong_results)
arch_results = test_results[test_results["true_label"] == "arch"]
arch_correct = arch_results["correct"].sum()

print(f"The baseline model misclassified {total_wrong} out of {total_test_images} test images.")
print(f"For the arch class, it correctly classified {arch_correct} out of {len(arch_results)} test images.")
print("The strongest limitation of the baseline is its poor recognition of the smaller arch class.")
print("This supports moving from HOG + Linear SVM to a stronger image-based model such as a CNN or transfer learning model.")

## Stop before the CNN model

Pause here and review the error-analysis results.

The next notebook should build a stronger image-based classifier and compare it against this baseline.